In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_id = "mystic-ai/Mistral-7B-Instruct-v0.2-GGUF" # O una versione simile leggera
# Per ora, usiamo una configurazione che non faccia esplodere la RAM:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map="auto"
)

# Creiamo la pipeline di generazione
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
def generate_answer(query, retrieved_chunks):
    # Uniamo i chunk in un unico blocco di contesto
    context = "\n\n".join(retrieved_chunks)
    
    # Costruiamo il prompt (istruzione + contesto + domanda)
    prompt = f"""
    You are a financial analyst expert in SEC filings. 
    Use the following pieces of context from a 10-K report to answer the user's question.
    If you don't know the answer based on the context, just say that you don't know, don't try to make up an answer.

    CONTEXT:
    {context}

    QUESTION: 
    {query}

    ANSWER:
    """
    
    # Generazione della risposta
    outputs = hf_pipeline(prompt, max_new_tokens=256, do_sample=True, temperature=0.1)
    return outputs[0]["generated_text"].split("ANSWER:")[-1].strip()

In [ ]:
from src.retrieval.retriever import Retriever

# 1. Inizializza il "Cercatore" (quello che abbiamo fatto ieri)
index_file = "data/embeddings/tsla_index.bin"
chunks_file = "data/chunks/tsla_10k_2025_chunks.json"
retriever = Retriever(index_file, chunks_file)

# 2. Definiamo la domanda
query = "What are the specific risks Tesla mentions about global supply chain and raw materials?"

# 3. Fase di Retrieval (Recupero informazioni)
print("--- Cercando nei documenti... ---")
context_chunks = retriever.search(query, k=4)

# 4. Fase di Generation (L'LLM risponde)
print("--- L'LLM sta scrivendo la risposta... ---")
risposta_finale = generate_answer(query, context_chunks)

print("\n" + "="*50)
print(f"DOMANDA: {query}")
print("="*50)
print(f"RISPOSTA IA:\n\n{risposta_finale}")
print("="*50)

ImportError: attempted relative import with no known parent package